# Gemma-3-1B Activation Precomputation (GPU T4)

Этот ноутбук предназначен для предварительного вычисления активаций (hidden states) перед слоем Titans (слой 23) модели Gemma-3-1B. Это позволяет значительно ускорить обучение TitansBlock, так как графу JAX не нужно просчитывать первые 23 слоя Gemma при каждом шаге.

**Особенности:**
- Оптимизировано для GPU T4 (Colab Free/Pro).
- Автоматическая выгрузка шардов в HuggingFace Hub.
- Поддержка докачки (resume) при обрыве сессии.

In [1]:
# 1. Установка зависимостей
# !pip install -q --upgrade "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q git+https://github.com/google-deepmind/gemma.git
!pip install -q flax==0.12.5 optax==0.2.6 typeguard==4.4.1 datasets ml_dtypes huggingface_hub orbax-checkpoint

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 91.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.6/738.6 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.3/216.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.8/621.8 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [2]:
# 2. Аутентификация HuggingFace
from huggingface_hub import login
from google.colab import userdata
import os

login(userdata.get('HF_TOKEN'))

In [3]:
# 3. Загрузка чекпоинта Gemma-3-1B
!mkdir -p gemma3_1b_ckpt
# !gsutil -m cp -r gs://gemma-data/checkpoints/gemma3-1b-it ./gemma3_1b_ckpt

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://gemma-data/checkpoints/gemma3-1b-it/_CHECKPOINT_METADATA...
Copying gs://gemma-data/checkpoints/gemma3-1b-it/_METADATA...
Copying gs://gemma-data/checkpoints/gemma3-1b-it/commit_success.txt...
Copying gs://gemma-data/checkpoints/gemma3-1b-it/d/a1daed4f231347e2fd0cb5a1b54f1561...
Copying gs://gemma-data/checkpoints/gemma3-1b-it/descriptor/descriptor.pbtxt...
Copying gs://gemma-data/checkpoints/gemma3-1b-it/descriptor/uuid-5849594c-3575-4c4d-b188-50de681c9115...
Copying gs://gemma-data/checkpoints/gemma3-1b-it/manifest.ocdbt...
Copying gs://gemma-data/checkpoints/gemma3-1b-it/ocdbt.process_0/d/17704b7997342c02051f12cc435d63cc...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster

In [16]:
# 4. Конфигурация
config = {
    "gemma_ckpt": "gm.ckpts.CheckpointPath.GEMMA3_1B_IT",
    "dataset_repo": "veriga/openwebtext-gemma3-tokenized-1024",
    "output_dir": "./activations_layer23-32",
    "target_layer": 23,
    "max_seq_len": 1024,
    "batch_size": 32,          # BS=4 безопасно для T4 (16GB)
    "dtype": "bfloat16",      # JAX на GPU умеет эмулировать bf16
    "save_dtype": "bfloat16",
    "max_examples": None,     # None = весь датасет
    "resume": True,
    "push_activations": True,
    "activations_repo": "veriga/openwebtext-gemma3-tokenized-1024-activations-layer23",
    "upload_batch": 64,       # Делать коммит в HF каждые 64 шарда
    "upload_workers": 2
}

os.makedirs(config["output_dir"], exist_ok=True)

In [17]:
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import jax
import jax.numpy as jnp
import numpy as np
from gemma import gm
from gemma.gm.nn import _modules
from datasets import load_dataset

def _build_block_kwargs(config, layer_idx):
    attn_type = config.attention_types[layer_idx]
    is_local = attn_type == _modules.AttentionType.LOCAL_SLIDING
    return dict(
        num_heads=config.num_heads,
        num_kv_heads=config.num_kv_heads,
        embed_dim=config.embed_dim,
        head_dim=config.head_dim,
        hidden_dim=config.hidden_dim,
        sliding_window_size=config.sliding_window_size,
        use_post_attn_norm=config.use_post_attn_norm,
        use_post_ffw_norm=config.use_post_ffw_norm,
        attn_logits_soft_cap=config.attn_logits_soft_cap,
        attn_type=attn_type,
        query_pre_attn_scalar=config.query_pre_attn_scalar(),
        transpose_gating_einsum=config.transpose_gating_einsum,
        use_qk_norm=config.use_qk_norm,
        rope_base_frequency=config.local_base_frequency if is_local else config.global_base_frequency,
        rope_scale_factor=config.local_scale_factor if is_local else config.global_scale_factor,
    )

def make_forward_to_layer(target_layer: int):
    # Ручной проход до нужного слоя, чтобы избежать бага return_hidden_states
    model = gm.nn.Gemma3_1B()
    config = model.config
    blocks = [_modules.Block(name=f'layer_{i}', **_build_block_kwargs(config, i)) for i in range(target_layer)]
    
    @jax.jit
    def forward_fn(params, tokens, masks):
        B, L = tokens.shape
        embedding_table = params['embedder']['input_embedding']
        x = embedding_table[tokens]
        x = x * jnp.sqrt(config.embed_dim).astype(x.dtype)
        
        positions = jnp.broadcast_to(jnp.arange(L)[None, :], (B, L))
        causal = jnp.tril(jnp.ones((L, L), dtype=jnp.bool_))
        attn_mask = causal[None, :, :] & masks[:, None, :].astype(jnp.bool_)
        
        for i, block in enumerate(blocks):
            _, x = block.apply({'params': params[f'layer_{i}']}, x, positions, None, attn_mask)
            
        return x
    return forward_fn

def dataset_to_batches(ds, batch_size, max_seq_len, max_examples=None):
    total = len(ds)
    if max_examples: total = min(total, max_examples)
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = ds[start:end]
        raw_tokens = batch.get("tokens") or batch.get("input_ids")
        
        batch_tokens, batch_masks = [], []
        for tokens in raw_tokens:
            t_arr = np.array(tokens[:max_seq_len], dtype=np.int32)
            orig_len = len(t_arr)
            pad_len = max_seq_len - orig_len
            if pad_len > 0: t_arr = np.pad(t_arr, (0, pad_len))
            m_arr = np.zeros(max_seq_len, dtype=np.int32)
            m_arr[:orig_len] = 1
            batch_tokens.append(t_arr)
            batch_masks.append(m_arr)
        yield np.stack(batch_tokens), np.stack(batch_masks)

def upload_shard_batch(shard_paths, repo_id, token, max_retries=3):
    from huggingface_hub import CommitOperationAdd, create_commit
    ops = [CommitOperationAdd(path_in_repo=os.path.basename(p), path_or_fileobj=p) for p in shard_paths]
    for attempt in range(max_retries):
        try:
            create_commit(repo_id=repo_id, operations=ops, 
                          commit_message=f"Add {len(shard_paths)} shards",
                          token=token, repo_type="dataset")
            return True
        except Exception as e:
            print(f"Upload error: {e}")
            time.sleep(10 * (2**attempt))
    return False

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
print("🔧 Загрузка параметров...")
params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
forward_fn = make_forward_to_layer(config["target_layer"])

print("📂 Загрузка датасета...")
ds = load_dataset(config["dataset_repo"],
                  split="train",
                  cache_dir="/content/drive/Shareddrives/shared_veriga/jax_cache"
                  )

start_shard = 0
meta_path = os.path.join(config["output_dir"], "metadata.json")
if config["resume"] and os.path.exists(meta_path):
    with open(meta_path) as f: start_shard = json.load(f).get("next_shard", 0)
    print(f"📋 Возобновление с шарда {start_shard}")

executor = ThreadPoolExecutor(max_workers=config["upload_workers"])
pending_shards = []
t_start = time.time()
shard_idx = start_shard

save_dtype = jnp.bfloat16 if config["save_dtype"] == "bfloat16" else np.float32

batch_gen = dataset_to_batches(ds, config["batch_size"], config["max_seq_len"], config["max_examples"])
for _ in range(start_shard): next(batch_gen, None)

print(f"🚀 Начало обработки (GPU: {jax.devices()[0].device_kind})...")
for batch_tokens, batch_masks in batch_gen:
    hidden = forward_fn(params, jnp.array(batch_tokens), jnp.array(batch_masks))
    hidden_np = np.asarray(hidden) * batch_masks[:, :, None].astype(np.float32)

    paths = []
    for suffix, data in [("", hidden_np.astype(save_dtype)), ("_tokens", batch_tokens), ("_masks", batch_masks)]:
        path = os.path.join(config["output_dir"], f"shard_{shard_idx:06d}{suffix}.npy")
        np.save(path, data)
        paths.append(path)

    if config["push_activations"]:
        pending_shards.extend(paths)
        if len(pending_shards) >= config["upload_batch"] * 3:
            executor.submit(upload_shard_batch, pending_shards[:], config["activations_repo"], hf_token)
            pending_shards = []

    shard_idx += 1
    if shard_idx % 10 == 0:
        elapsed = time.time() - t_start
        print(f"  Shard {shard_idx} | {shard_idx*config['batch_size']/elapsed:.1f} ex/s | {elapsed:.1f}s")
        with open(meta_path, "w") as f: json.dump({"next_shard": shard_idx}, f)

print("✅ Обработка завершена!")

🔧 Загрузка параметров...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_362/2799619711.py", line 2, in <cell line: 0>
    params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gemma/gm/ckpts/_checkpoint.py", line 232, in load_params
    metadata, path = _get_metadata_and_path(ckpt, path)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gemma/gm/ckpts/_checkpoint.py", line 497, in _get_metadata_and_path
    metadata = ckpt.metadata(path)
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/orbax/checkpoint/_src/checkpointers/checkpointer.py", line 339, in metadata
    checkpoint.step_metadata_file_path(directory)
  File "/usr/lo

TypeError: object of type 'NoneType' has no len()

In [22]:
!huggingface-cli upload --repo-type dataset veriga/openwebtext-gemma3-tokenized-1024-activations-layer23 ./activations_layer23-32



Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [ ]:
!hf upload-large-folder veriga/openwebtext-gemma3-tokenized-1024-activations-layer23 ./activations_layer23-32/ --repo-type dataset


A few things to keep in mind:
  - Repository limits still apply: https://huggingface.co/docs/hub/repositories-recommendations
  - Do not start several processes in parallel.
  - You can interrupt and resume the process at any time. The script will pick up where it left off except for partially uploaded files that would have to be entirely reuploaded.
  - Do not upload the same folder to several repositories. If you need to do so, you must delete the `./.cache/huggingface/` folder first.

Some temporary metadata will be stored under `./activations_layer23-32//.cache/huggingface`.
  - You must not modify those files manually.
  - You must not delete the `./.cache/huggingface/` folder while a process is running.
  - You can delete the `./.cache/huggingface/` folder to reinitialize the upload state when process is not running. Files will have to be hashed and preuploaded again, except for already committed files.

If the process output is too verbose, you can disable the progress bars wit